# 09 · 시계열 전처리와 추세·변동 분석

시각, 주기, 결측 간격을 먼저 확인한 뒤 이동 통계와 변화량을 계산합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## 시간형 변환과 정렬

**코드 → 코드 개념**: 문자열은 시간 연산 전에 `datetime`으로 바꿔야 한다.

**코드 사용법**: 설비별 시간순 표를 만든다.

In [ ]:
import pandas as pd
import numpy as np
raw = pd.DataFrame({"machine": ["A"] * 5,
                    "timestamp": ["2026-09-01 09:00:00", "2026-09-01 09:00:10", "2026-09-01 09:00:30", "2026-09-01 09:00:40", "2026-09-01 09:00:50"],
                    "vibration": [2.0, 2.2, 2.4, 3.1, 3.6]})
raw["timestamp"] = pd.to_datetime(raw["timestamp"], errors="coerce")
ts = raw.sort_values(["machine", "timestamp"]).set_index("timestamp")
print(ts.index.dtype, ts.head())

**같은 결과를 얻는 방법과 선택 이유**

- 날짜 형식이 일정하면 `format=`을 지정해 예상 밖 형식을 잡는다. `errors='coerce'`는 실패값을 `NaT`로 바꾸므로 건수를 확인한다.
- 실제 설비 ID가 여러 개면 설비별로 분리하거나 `groupby` 후 이동 통계를 계산한다.

## 간격 확인: asfreq와 resample

**코드 → 코드 개념**: `asfreq`는 시간격자를 만들고, `resample`은 구간 안의 관측을 집계한다.

**코드 사용법**: 10초 결측과 30초 평균을 확인한다.

In [ ]:
interval = ts.index.to_series().diff().dt.total_seconds()
grid = ts["vibration"].asfreq("10s")
half_minute = ts["vibration"].resample("30s").agg(["mean", "max", "count"])
print(interval, grid, half_minute, sep="\n")

**같은 결과를 얻는 방법과 선택 이유**

- 단순히 빠진 시각을 표시하려면 `asfreq`를 쓴다. 여러 측정값을 시간 구간별로 요약하려면 `resample`을 쓴다. 둘을 같은 것으로 보면 안 된다.
- 평균은 급등을 숨기므로 최대와 건수도 같이 본다.

## 결측 채우기: 이전 값과 시간 보간

**코드 → 코드 개념**: `ffill`은 직전 값 유지, `interpolate(method='time')`은 양끝 값 사이를 시간에 따라 추정한다.

**코드 사용법**: 한 칸만 채우고 어떤 위치를 채웠는지 기록한다.

In [ ]:
forward = grid.ffill(limit=1)
linear_time = grid.interpolate(method="time", limit=1, limit_area="inside")
imputed = grid.isna() & linear_time.notna()
print(pd.DataFrame({"raw": grid, "ffill": forward, "time_interpolate": linear_time, "imputed": imputed}))

**같은 결과를 얻는 방법과 선택 이유**

- 상태가 다음 측정까지 유지된다고 볼 때 `ffill`을 쓴다. 연속 물리량이 부드럽게 변한다고 볼 때 시간 보간을 고려한다.
- 운전 중지·정비·긴 결측 구간은 채우면 실제 이벤트가 사라질 수 있다. 채운 값은 플래그로 남긴다.

## 이동평균과 이동표준편차

**코드 → 코드 개념**: 이동평균은 국소 추세, 이동표준편차는 국소 변동성을 보여 준다.

**코드 사용법**: 행 개수 창과 시간 길이 창을 비교한다.

In [ ]:
ts["ma3"] = ts["vibration"].rolling(3, min_periods=3).mean()
ts["std3"] = ts["vibration"].rolling(3, min_periods=3).std()
ts["ma30s"] = ts["vibration"].rolling("30s").mean()
print(ts)

**같은 결과를 얻는 방법과 선택 이유**

- `rolling(3)`은 최근 3개 관측, `rolling('30s')`는 최근 30초다. 측정 간격이 불규칙하면 두 결과가 달라진다.
- `min_periods`를 작게 하면 초기 값도 나오지만 표본 수가 적어 비교가 불안정하다.

## 차분, 변화율, 설비 상태

**코드 → 코드 개념**: `diff`는 이전 값과의 차이, `pct_change`는 상대 변화율이다.

**코드 사용법**: 진동의 급변 후보를 찾는다.

In [ ]:
ts["diff"] = ts["vibration"].diff()
ts["pct"] = ts["vibration"].pct_change()
ts["jump"] = ts["diff"].abs() >= 0.5
print(ts[["vibration", "diff", "pct", "jump"]])

**같은 결과를 얻는 방법과 선택 이유**

- 절대 변화량이 중요하면 `diff`, 단위와 기준값이 다른 설비를 비교하면 `pct_change`가 유용하다. 직전 값이 0에 가깝다면 변화율이 폭발하므로 주의한다.
- 급변 기준은 정상 기간의 변화량 분포와 정비 이력을 함께 보고 정한다.

## 원본 학습 자료

[`4. Summary/08_matplotlib/03_TimeSeries_Data.ipynb`](../4.%20Summary/08_matplotlib/03_TimeSeries_Data.ipynb), [`04_TimeSeries_Preprocessing.ipynb`](../4.%20Summary/08_matplotlib/04_TimeSeries_Preprocessing.ipynb), [`05_Equipment_State_Analysis.ipynb`](../4.%20Summary/08_matplotlib/05_Equipment_State_Analysis.ipynb)